## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date = '20240122'
end_date = '20240122'

In [7]:
## pricing

pricing = f"""

    SELECT 
        yyyymmdd,
        platform,
        user_id,
        fare_estimate_id,
        api_context
        -- SUBSTR(quarter_hour, 1,2) hour
    FROM
        pricing.fare_estimates_enriched
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND platform IN ('android', 'ios')
    GROUP BY 1,2,3,4,5
"""

df_pricing = pd.read_sql(pricing, connection)
df_pricing.head(3)

,yyyymmdd,platform,user_id,fare_estimate_id,api_context
0,20240122,android,5cf6587bc0b26018f5d8ca45,65addf98b536308a04e99ba5,/fare/estimate
1,20240122,android,61a4ff8bf3de6f40063d95f6,65ae9b9fca2555d829239a1f,/fare/estimate
2,20240122,ios,5a2d148b31e7165d849fa25e,65ae4f7766ce332ef0bd65ca,/fare/estimate


In [8]:
## clevertap
clevertap = f"""

    SELECT 
        yyyymmdd,
        user_id,
        event_props_app_variant app_variant,
        ct_app_version app_version,
        fare_estimate_id
        -- SUBSTR(quarter_hour, 1,2) hour
    FROM
        canonical.clevertap_customer_fare_estimate ct_fe
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    GROUP BY 1,2,3,4,5
"""

df_clevertap = pd.read_sql(clevertap, connection)
df_clevertap.head(3)

,yyyymmdd,user_id,app_variant,app_version,fare_estimate_id
0,20240122,62420cb4b9a21efeced0c45d,None,3.1.1,None
1,20240122,652fce4c1556073579715ea2,v8,8.14.0,65adcb292db45f604c5c2b9c
2,20240122,61be08334fc413f14c35abbe,v8,8.13.0,65ae81ee4998dc72f1ab8236


In [9]:
df_merge = pd.merge(df_pricing,
                    df_clevertap,
                    how = 'left',
                    on = ['yyyymmdd', 'user_id']
                   )
df_merge

,yyyymmdd,platform,user_id,fare_estimate_id_x,api_context,app_variant,app_version,fare_estimate_id_y
0,20240122,android,5cf6587bc0b26018f5d8ca45,65addf98b536308a04e99ba5,/fare/estimate,v8,8.14.0,65adeb21714627fe8c5114be
1,20240122,android,5cf6587bc0b26018f5d8ca45,65addf98b536308a04e99ba5,/fare/estimate,v8,8.14.0,65adeb0e482e1b2f792316c2
2,20240122,android,5cf6587bc0b26018f5d8ca45,65addf98b536308a04e99ba5,/fare/estimate,v8,8.14.0,65addf98b536308a04e99ba5
3,20240122,android,5cf6587bc0b26018f5d8ca45,65addf98b536308a04e99ba5,/fare/estimate,v8,8.14.0,65adf444314d1ba6fe539021
4,20240122,android,5cf6587bc0b26018f5d8ca45,65addf98b536308a04e99ba5,/fare/estimate,v8,8.14.0,65addf7da8f64d5cec184ef0
...,...,...,...,...,...,...,...,...
30720328,20240122,android,65ab96b218713d4e477d762b,65adf2d3f8632676a93040cb,/fare/estimate,v8,8.14.0,65adf747714627a9dd52121b
30720329,20240122,android,65ab96b218713d4e477d762b,65adf2d3f8632676a93040cb,/fare/estimate,v8,8.14.0,65adf2ec45b15775349aa511
30720330,20240122,android,65ab96b218713d4e477d762b,65adf2d3f8632676a93040cb,/fare/estimate,v8,8.14.0,65adf768bf5c915243dee5b7
30720331,20240122,android,65ab96b218713d4e477d762b,65adf2d3f8632676a93040cb,/fare/estimate,v8,8.14.0,65adf616714627bd3551fc73
